# Notebook 01 — Exploratory Data Analysis
**Project:** Predictive Maintenance System — Microsoft Azure Dataset  
**Author:** Abrar Ahmed  
**Phase:** 0 — Understand the data before touching a model

---
### What we do in this notebook
1. Load all 5 raw tables and understand their structure
2. Merge them into a single analysis-ready dataframe
3. Explore sensor distributions, failure patterns, and class imbalance
4. Produce the key plots that will go into the README

**Rule:** No model code in this notebook. EDA only. Understanding the data first is what separates engineers from notebook hackers.

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting style settings
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

# Paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

print('Libraries loaded successfully')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

## 1. Load the five raw tables

This dataset has 5 tables that together describe a fleet of 100 machines over 1 year:

| Table | What it contains | Granularity |
|---|---|---|
| `telemetry` | 4 sensor readings per machine | Hourly |
| `errors` | Error code events | Event-based |
| `failures` | Component failure events (our TARGET) | Event-based |
| `maint` | Maintenance/replacement events | Event-based |
| `machines` | Machine model and age | One row per machine |

In [ ]:
# Load all tables
# Note: we parse datetime columns immediately — always do this on load, not later
telemetry = pd.read_csv(DATA_RAW / 'PdM_telemetry.csv', parse_dates=['datetime'])
errors     = pd.read_csv(DATA_RAW / 'PdM_errors.csv',   parse_dates=['datetime'])
failures   = pd.read_csv(DATA_RAW / 'PdM_failures.csv', parse_dates=['datetime'])
maint      = pd.read_csv(DATA_RAW / 'PdM_maint.csv',    parse_dates=['datetime'])
machines   = pd.read_csv(DATA_RAW / 'PdM_machines.csv')

print('=== TABLE SHAPES ===')
for name, df in [('telemetry', telemetry), ('errors', errors),
                  ('failures', failures), ('maint', maint), ('machines', machines)]:
    print(f'  {name:<12}: {df.shape[0]:>8,} rows  x  {df.shape[1]} cols')

## 2. Understand each table individually



In [ ]:
# ── TELEMETRY ────────────────────────────────────────────────────────────────
print('=== TELEMETRY ===')
print(telemetry.head(10).to_string())
print('\nDate range:', telemetry['datetime'].min(), '→', telemetry['datetime'].max())
print('Machines :', telemetry['machineID'].nunique())
print('\nNull counts:')
print(telemetry.isnull().sum())

In [ ]:
# ── FAILURES ─────────────────────────────────────────────────────────────────
# This is our target variable — understand it deeply
print('=== FAILURES ===')
print(failures.head(20).to_string())
print('\nFailure component counts:')
print(failures['failure'].value_counts())
print('\nTotal failure events:', len(failures))
print('Machines with at least one failure:', failures['machineID'].nunique())

In [ ]:
# ── ERRORS ───────────────────────────────────────────────────────────────────
print('=== ERRORS ===')
print(errors.head(10).to_string())
print('\nError type counts:')
print(errors['errorID'].value_counts())

In [ ]:
# ── MACHINES ─────────────────────────────────────────────────────────────────
print('=== MACHINES ===')
print(machines.head(10).to_string())
print('\nModel distribution:')
print(machines['model'].value_counts())
print('\nAge distribution (years):')
print(machines['age'].describe().round(1))

## 3. Visualise sensor distributions

The four sensors — `volt`, `rotate`, `pressure`, `vibration` — are what the model will learn from.  
We need to understand their ranges, distributions, and any obvious outliers.

In [ ]:
sensors = ['volt', 'rotate', 'pressure', 'vibration']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

colors = ['#185FA5', '#1D9E75', '#BA7517', '#A32D2D']

for i, (sensor, color) in enumerate(zip(sensors, colors)):
    axes[i].hist(telemetry[sensor], bins=80, color=color, alpha=0.75, edgecolor='none')
    axes[i].set_title(f'{sensor.capitalize()} distribution', fontweight='bold')
    axes[i].set_xlabel(sensor)
    axes[i].set_ylabel('Count')
    # Add mean line
    mean_val = telemetry[sensor].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
                    label=f'mean={mean_val:.1f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Sensor value distributions (8.7M readings)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/sensor_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved to data/processed/sensor_distributions.png')

In [ ]:
# Summary statistics for the sensors
print('=== SENSOR SUMMARY STATISTICS ===')
print(telemetry[sensors].describe().round(2).to_string())

## 4. Visualise a single machine's sensor timeline

This is the most important plot for understanding what we're predicting.  
We pick one machine, plot its sensors over time, and overlay failure events.

In [ ]:
# Pick a machine that has at least 2 failures — more interesting to visualise
machines_with_failures = failures.groupby('machineID').size()
interesting_machine = machines_with_failures[machines_with_failures >= 2].index[0]
print(f'Selected machine: {interesting_machine}')
print(f'Failures: {machines_with_failures[interesting_machine]}')

# Get its telemetry and failures
m_tel = telemetry[telemetry['machineID'] == interesting_machine].copy()
m_fail = failures[failures['machineID'] == interesting_machine].copy()

print(f'Telemetry rows: {len(m_tel)}')
print(f'Failure events: {len(m_fail)}')
print(m_fail)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)

colors = ['#185FA5', '#1D9E75', '#BA7517', '#A32D2D']

for i, (sensor, color) in enumerate(zip(sensors, colors)):
    axes[i].plot(m_tel['datetime'], m_tel[sensor],
                 color=color, linewidth=0.6, alpha=0.8)
    axes[i].set_ylabel(sensor, fontweight='bold')

    # Overlay failure events as vertical red lines
    for _, row in m_fail.iterrows():
        axes[i].axvline(row['datetime'], color='red', linewidth=1.5,
                        linestyle='--', alpha=0.8,
                        label='failure' if i == 0 else '')

    axes[i].grid(True, alpha=0.3)

# Add legend only on top plot
axes[0].legend(loc='upper right')
axes[0].set_title(f'Machine {interesting_machine} — sensor readings with failure events',
                  fontweight='bold', fontsize=12)
axes[-1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('../data/processed/machine_timeline.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved — this goes in the README')

## 5. Class imbalance analysis

This is critical for model design. If failures are rare (they will be), we need strategies like SMOTE.  
We frame the prediction problem as: **will this machine fail in the next 24 hours?**

In [ ]:
# Count total telemetry hours vs failure hours
total_hours = len(telemetry)

# For each failure event, we label the 24 hours BEFORE it as positive
# This is the label engineering step we'll do properly in notebook 02
# Here we just estimate the imbalance ratio

# Conservative estimate: each failure event covers 24 hours = 24 rows per machine
positive_hours_estimate = len(failures) * 24
imbalance_ratio = total_hours / positive_hours_estimate

print(f'Total telemetry rows (hours)   : {total_hours:>10,}')
print(f'Failure events total            : {len(failures):>10,}')
print(f'Estimated positive class rows   : {positive_hours_estimate:>10,}')
print(f'Estimated negative class rows   : {total_hours - positive_hours_estimate:>10,}')
print(f'Imbalance ratio (neg:pos)       : {imbalance_ratio:>10.1f}x')
print()
print('→ This is significant imbalance. We MUST use SMOTE or class weighting.')
print('→ We will NOT use accuracy as our metric. We use F1, Precision, Recall, ROC-AUC.')

In [ ]:
# Failure distribution by component
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: failure counts by component
fail_counts = failures['failure'].value_counts()
bars = axes[0].bar(fail_counts.index, fail_counts.values,
                   color='#A32D2D', alpha=0.8, edgecolor='none')
axes[0].set_title('Failure events by component', fontweight='bold')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Number of failure events')
for bar, val in zip(bars, fail_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Right: failures per machine histogram
fail_per_machine = failures.groupby('machineID').size()
axes[1].hist(fail_per_machine, bins=20, color='#185FA5', alpha=0.8, edgecolor='none')
axes[1].set_title('Failures per machine (over 1 year)', fontweight='bold')
axes[1].set_xlabel('Number of failures')
axes[1].set_ylabel('Number of machines')
axes[1].axvline(fail_per_machine.mean(), color='red', linestyle='--',
                label=f'mean={fail_per_machine.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/failure_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Sensor correlation analysis

Understanding which sensors are correlated tells us about redundancy and what features matter.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

corr = telemetry[sensors].corr()
mask = np.triu(np.ones_like(corr), k=1)  # upper triangle mask

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            square=True, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Sensor correlation matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/sensor_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Do sensors behave differently before failures?

This is the key question. If sensors show anomalous behaviour BEFORE a failure, our model has signal to learn from.  
We compare sensor means in the 24h window before a failure vs normal operation.

In [ ]:
# Build failure windows: for each failure, get telemetry in the 24h before it
pre_failure_windows = []

for _, fail_row in failures.iterrows():
    machine_id = fail_row['machineID']
    fail_time  = fail_row['datetime']
    window_start = fail_time - pd.Timedelta(hours=24)

    window = telemetry[
        (telemetry['machineID'] == machine_id) &
        (telemetry['datetime'] >= window_start) &
        (telemetry['datetime'] < fail_time)
    ].copy()
    window['label'] = 'pre_failure'
    pre_failure_windows.append(window)

pre_failure_df = pd.concat(pre_failure_windows, ignore_index=True)

# Sample the same number of rows from normal operation
normal_df = telemetry[~telemetry.index.isin(pre_failure_df.index)].sample(
    n=len(pre_failure_df), random_state=42
).copy()
normal_df['label'] = 'normal'

combined = pd.concat([pre_failure_df, normal_df], ignore_index=True)
print(f'Pre-failure rows : {len(pre_failure_df):,}')
print(f'Normal rows      : {len(normal_df):,}')

In [ ]:
# Compare sensor distributions: normal vs pre-failure
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, sensor in enumerate(sensors):
    for label, color, alpha in [('normal', '#185FA5', 0.6),
                                  ('pre_failure', '#A32D2D', 0.7)]:
        data = combined[combined['label'] == label][sensor]
        axes[i].hist(data, bins=60, color=color, alpha=alpha,
                     label=label, density=True, edgecolor='none')
    axes[i].set_title(f'{sensor.capitalize()}: normal vs pre-failure', fontweight='bold')
    axes[i].legend()
    axes[i].set_xlabel(sensor)
    axes[i].set_ylabel('Density')

plt.suptitle('Sensor distributions: normal operation vs 24h before failure',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/sensor_prefailure_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('If the distributions differ → our model has real signal. If identical → hard problem.')

## 8. Error patterns

Error events (separate from failures) may be leading indicators. We check if errors cluster before failures.

In [ ]:
# Fast vectorized version — same result, no slow loop
# Merge errors onto failures by machineID, then filter by time window

errors_tagged = errors.copy()
failures_tagged = failures.copy()

# Cross-join approach: merge all errors with all failures on machineID
merged = failures_tagged.merge(errors_tagged, on='machineID', suffixes=('_fail', '_err'))

# Keep only errors that fall within 24h BEFORE the failure
in_window = (
    (merged['datetime_err'] >= merged['datetime_fail'] - pd.Timedelta(hours=24)) &
    (merged['datetime_err'] <  merged['datetime_fail'])
)
errors_in_window = merged[in_window].groupby(merged[in_window].index).size()

# Count per failure event
errors_before_failure = (
    merged[in_window]
    .groupby(['machineID', 'datetime_fail'])
    .size()
    .reindex(pd.MultiIndex.from_frame(failures_tagged[['machineID', 'datetime']]), fill_value=0)
    .values
)

# Random baseline — vectorized
np.random.seed(42)
random_times = pd.date_range(telemetry['datetime'].min(),
                              telemetry['datetime'].max(), freq='24h')
sample_times = pd.Series(np.random.choice(random_times, size=len(failures), replace=True))
random_machines = np.random.choice(telemetry['machineID'].unique(),
                                    size=len(failures), replace=True)

random_df = pd.DataFrame({'machineID': random_machines, 'random_time': sample_times})
merged_rand = random_df.merge(errors_tagged, on='machineID')
in_rand_window = (
    (merged_rand['datetime'] >= merged_rand['random_time']) &
    (merged_rand['datetime'] <  merged_rand['random_time'] + pd.Timedelta(hours=24))
)
random_errors = merged_rand[in_rand_window].groupby(merged_rand[in_rand_window].index).size()
random_mean = len(merged_rand[in_rand_window]) / len(failures)

print('=== ERRORS IN 24H WINDOW ===')
print(f'Before failure  — mean: {errors_before_failure.mean():.2f}')
print(f'Random window   — mean: {random_mean:.2f}')
print()
print('→ If errors_before_failure mean >> random mean, errors are a useful feature.')

## 9. Machine age and model analysis

In [ ]:
# Merge failures with machine metadata to check if age/model affects failure rate
failures_with_meta = failures.merge(machines, on='machineID', how='left')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Failure count by model
model_fail = failures_with_meta['model'].value_counts()
axes[0].bar(model_fail.index, model_fail.values, color='#185FA5', alpha=0.8, edgecolor='none')
axes[0].set_title('Failure count by machine model', fontweight='bold')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Failure count')

# Age distribution of failed machines
axes[1].hist(failures_with_meta['age'], bins=15, color='#BA7517', alpha=0.8, edgecolor='none')
axes[1].set_title('Age of machines at failure', fontweight='bold')
axes[1].set_xlabel('Machine age (years)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/processed/machine_meta_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Save the merged base dataframe

We create one clean merged table that notebook 02 will use as its starting point.  
We save it as **parquet** — much faster than CSV for large dataframes.

In [ ]:
# Merge telemetry with machine metadata (every row gets machine model + age)
base_df = telemetry.merge(machines, on='machineID', how='left')

# Sort by machine and time — critical for time series operations later
base_df = base_df.sort_values(['machineID', 'datetime']).reset_index(drop=True)

print('=== BASE DATAFRAME ===')
print(f'Shape: {base_df.shape}')
print(f'Columns: {base_df.columns.tolist()}')
print(f'Dtypes:')
print(base_df.dtypes)
print(f'\nNull counts:')
print(base_df.isnull().sum())
print(f'\nFirst rows:')
print(base_df.head(5).to_string())

In [ ]:
# Save as parquet — fast columnar format, much better than CSV for ML pipelines
base_df.to_parquet(DATA_PROCESSED / 'base_telemetry.parquet', index=False)

# Also save error and failure tables as parquet for notebook 02
errors.to_parquet(DATA_PROCESSED / 'errors.parquet', index=False)
failures.to_parquet(DATA_PROCESSED / 'failures.parquet', index=False)
maint.to_parquet(DATA_PROCESSED / 'maint.parquet', index=False)

print('Saved to data/processed/')
print('  - base_telemetry.parquet')
print('  - errors.parquet')
print('  - failures.parquet')
print('  - maint.parquet')

## 11. EDA Summary

Before moving to notebook 02, write down your answers to these questions.  
If you cannot answer them, re-read the cells above.

**Questions:**
1. How many machines are in the fleet?
2. How long is the observation period?
3. How many total failure events are there, and which component fails most often?
4. What is the approximate imbalance ratio (negative:positive class)?
5. Do the sensor distributions look different before failure vs during normal operation?
6. Which sensor appears most correlated with others?
7. Are errors a useful signal (do they cluster before failures)?

**Write your answers here (double-click to edit this cell):**

1. 100
2. 2015-01-01 06:00:00 → 2016-01-01 06:00:00
3. 761 total failures and component 2 fails the most
4. neg:positive approximately 48.0
5. yes
6. none of the sensors are correlated
7. yes

---
### Next step
Once you can answer all 7 questions, open **02_features.ipynb** — we build the scalable feature engineering pipeline with PySpark.